## Setup and Imports


In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from glob import glob

## Load Dataset


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REAL_PATH = "/content/drive/MyDrive/Data/real"
FAKE_PATH = "/content/drive/MyDrive/Data/fake"

## Image loader

```
# This is formatted as code
```




In [ ]:
IMG_SIZE = 224

def load_images(folder, label, max_images=500):
    images = []
    labels = []

    files = os.listdir(folder)[:max_images]

    for file in files:
        path = os.path.join(folder, file)
        img = cv2.imread(path)

        if img is None:
            continue

        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        img = img / 255.0

        images.append(img)
        labels.append(label)

    return images, labels

# Preprocess Images

In [ ]:
uadfv_path_real = "/content/drive/My Drive/CS445/CS445 Final Project/Data/UADFV_sample/real/" #note: change this to fit how your drive is organized
#add any other datapath you're using here
#can also use the paths in the load images cell i think

IMG_SIZE = 256

def preprocess_image(image_path):
    img = cv2.imread(image_path)

    if img is None:
        return None

    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    img = img.astype(np.float32) / 255.0

    return img

In [ ]:
image_paths = glob(uadfv_path_real + "/**/*.png", recursive=True)

processed_images = []

for path in image_paths:
    img = preprocess_image(path)

    if img is not None:
        processed_images.append(img)

processed_images = np.array(processed_images)

#repeat for any other datasets we have

## Load both classes




In [ ]:
real_images, real_labels = load_images(REAL_PATH, 0)
fake_images, fake_labels = load_images(FAKE_PATH, 1)

X_images = real_images + fake_images
y_labels = real_labels + fake_labels

print("Total images:", len(X_images))

## FFT Function


In [ ]:
def compute_fft(image):
    fft = np.fft.fft2(image)
    fft_shift = np.fft.fftshift(fft)
    magnitude = np.log(np.abs(fft_shift) + 1)
    return magnitude

## Visualize FFT

In [ ]:
def show_fft(image):
    fft_mag = compute_fft(image)

    plt.figure(figsize=(10,4))

    plt.subplot(1,2,1)
    plt.title("Original")
    plt.imshow(image, cmap='gray')

    plt.subplot(1,2,2)
    plt.title("FFT Magnitude")
    plt.imshow(fft_mag, cmap='gray')

    plt.show()

## Test Visualization

In [ ]:
how_fft(X_images[0])
show_fft(X_images[len(real_images)])  # fake example

## Feature Extraction

In [ ]:
def extract_features(magnitude):
    h, w = magnitude.shape

    # center = low frequency
    center = magnitude[h//4:3*h//4, w//4:3*w//4]

    # edges = high frequency
    edges = magnitude.copy()
    edges[h//4:3*h//4, w//4:3*w//4] = 0

    mean = np.mean(magnitude)
    std = np.std(magnitude)

    low_energy = np.sum(center)
    high_energy = np.sum(edges)

    return [mean, std, low_energy, high_energy]

## Build Feature Matrix

In [ ]:
X_features = []

for img in X_images:
    fft_mag = compute_fft(img)
    features = extract_features(fft_mag)
    X_features.append(features)

X_features = np.array(X_features)
y_labels = np.array(y_labels)

print("Feature shape:", X_features.shape)

## Train Logistic Regression

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_features, y_labels, test_size=0.2, random_state=42
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

## Evaluate Model

In [ ]:
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

## Visualization (for report)

In [ ]:
# Compare feature distributions
import seaborn as sns

real_feats = X_features[y_labels == 0]
fake_feats = X_features[y_labels == 1]

plt.figure(figsize=(10,5))
sns.histplot(real_feats[:,3], color='blue', label='Real', kde=True)
sns.histplot(fake_feats[:,3], color='red', label='Fake', kde=True)
plt.legend()
plt.title("High Frequency Energy Distribution")
plt.show()